[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HernanBustamanteG/lapis26-p3-corrientes/blob/main/notebooks/09_videos.ipynb)

# 09 · Videos

La formación de una corriente larga durante 5 Gyr en los cuatro modelos. Las instantáneas salen del propio
generador de `gala` (`output_every`, integrador DOP853, archivo HDF5), como sugiere el enunciado; la corriente
final coincide con la del catálogo (integrador por defecto) a ~10⁻⁴ kpc.

- **Académico** (1920×1080, 30 cuadros/s, H.264): 2×2 paneles en el plano orbital inicial con la órbita de la LMC, y
  la inclinación del polo medio respecto del polo inicial en función del tiempo.
- **Prensa** (1920×1080): densidad luminosa de la corriente con la LMC sobre la misma corriente sin ella, el disco
  y la LMC.
- GIF de cada uno (960 px, 15 cuadros/s) para diapositivas.

Tiempo en Colab: ~5 min de simulación y ~5 min de video.

In [1]:
# Celda 1: entorno. En Colab clona el repositorio, instala versiones fijas y monta Google Drive (las salidas
# quedan en MyDrive/lapis26_p3/salidas). Si numpy cambia de version, la sesion se reinicia UNA vez:
# vuelve a ejecutar esta celda. En una maquina propia solo comprueba versiones.
import os, sys, subprocess
REPO_URL = "https://github.com/HernanBustamanteG/lapis26-p3-corrientes.git"
USAR_DRIVE = True
if "google.colab" in sys.modules:
    RAIZ = "/content/lapis26-p3-corrientes"
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "-q", REPO_URL, RAIZ], check=True)
else:
    RAIZ = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, RAIZ)
os.chdir(RAIZ)
from p3streams import entorno
if entorno.en_colab():
    entorno.preparar_colab(RAIZ, montar_drive=USAR_DRIVE)
entorno.resumen();

Python 3.13.16
  numpy       2.2.6
  scipy       1.18.1
  astropy     8.0.2
  gala        1.12.0


  matplotlib  3.11.2
  pandas      3.1.0


  healpy      1.20.1
  galstreams  1.2.1
Salidas en: salidas
ffmpeg: /usr/bin/ffmpeg | CPU: 2
Entorno OK


In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
import matplotlib.pyplot as plt
from p3streams import config as C, util as U, estilo as E
E.usar()
SAL = U.carpeta_nueva("09_videos")          # carpeta nueva de este notebook (nada se pisa)
FUENTE = "notebooks/09_videos.ipynb, p3streams " + C.VERSION + ", " + U.ahora()
print("Salidas de este notebook:", SAL)

Salidas de este notebook: salidas/09_videos_20261027_211314


In [3]:
import time
from p3streams import potenciales as P, videos as V, catalogo as Ca
sis = P.integrar_sistema()
FAM = Ca.FAMILIAS["larga"]
FILA = dict(id="video_larga_polo_y", polo=[0.0, -1.0, 0.0], r_apo=FAM["r_apo"], m_prog=FAM["m_prog"], f=0.5,
            psi_deg=0.0, t_start=C.T_START)
t0 = time.time()
RUTAS = V.simular(FILA, sis, os.path.join(SAL, "instantaneas"), output_every=10)
SN = {m: V.leer(r) for m, r in RUTAS.items()}
print("Instantaneas: %d cuadros x %d estrellas, %.0f s" % (len(SN["A"]["t"]), SN["A"]["pos"].shape[1], time.time() - t0))

Instantaneas: 501 cuadros x 5002 estrellas, 665 s


In [4]:
from p3streams import corrientes as Co
x0, v0 = Co.condiciones_iniciales(FILA["polo"], FILA["r_apo"], FILA["f"], FILA["psi_deg"])
PROC = "notebooks/09_videos.ipynb, p3streams %s, %s; stream r_apo 60 kpc, 5e7 Msun, 5 Gyr" % (C.VERSION, U.ahora())
t0 = time.time()
MP4A = V.academico(SN, sis, FILA["polo"], x0, os.path.join(SAL, "video_academico.mp4"),
                   titulo="Stream formation in the four models (MW frame)", procedencia=PROC)
print("academico: %.0f s" % (time.time() - t0)); t0 = time.time()
MP4P = V.prensa(SN["D"], SN["A"], sis, FILA["polo"], x0, os.path.join(SAL, "video_prensa.mp4"),
                titulo="How the Large Magellanic Cloud bends a stream of stars", procedencia=PROC)
print("prensa: %.0f s" % (time.time() - t0))
for mp4 in (MP4A, MP4P):
    V.a_gif(mp4, mp4[:-4] + ".gif")
for f in sorted(os.listdir(SAL)):
    if f.endswith((".mp4", ".gif")):
        print("%-24s %6.1f MB" % (f, os.path.getsize(os.path.join(SAL, f)) / 1e6))

academico: 53 s


prensa: 165 s


video_academico.gif         5.3 MB
video_academico.mp4         9.1 MB
video_prensa.gif            6.1 MB
video_prensa.mp4            3.0 MB


In [5]:
from IPython.display import Video, display
display(Video(MP4A, embed=False, width=720))
display(Video(MP4P, embed=False, width=720))